# PART 2: Training


In [9]:
import torch
import numpy as np
import os
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms,models
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from typing import Tuple, Dict, Optional
from torch.utils.data import WeightedRandomSampler, Dataset

# --- Configuration ---
CONFIG = {
    "SEED": 42,
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 2,
    "IMG_SIZE": 224,
    "DATA_DIR": "data"
}

def set_seed(seed: int = 42) -> None:
    """
    Sets the random seed for reproducibility across PyTorch and NumPy.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    print(f"🔒 Random Seed set to: {seed}")



### 1. Define Augmentations




In [2]:
# Justification: Resize to 224 for standard models.
# HorizontalFlip and Rotation account for different camera angles in the field.

def get_transforms(img_size: int = 224) -> Dict[str, transforms.Compose]:
    """
    Returns a dictionary of transforms for 'train' and 'val/test' phases.
    Includes augmentation for training to handle field conditions.
    """
    stats = {
        "mean": [0.485, 0.456, 0.406],
        "std":  [0.229, 0.224, 0.225]
    }
    return {
        "train": transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(**stats)
        ]),
        "val": transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(**stats)
        ])
    }

def get_weighted_sampler(dataset: Dataset) -> WeightedRandomSampler:
  """
  Calculates sample weights to balance the class distribution.
  Uses 'WeightedRandomSampler' to oversample rare classes.
  """
  # 1. Access labels safely
  # Note: Flowers102 stores labels in ._labels (list of ints)
  if hasattr(dataset, '_labels'):
      # Convert to tensor for bincount operations
      targets = torch.tensor(dataset._labels)
  else:
      # Fallback for generic datasets (slower)
      targets = torch.tensor([label for _, label in dataset])

  # 2. Calculate class counts
  class_counts = torch.bincount(targets)

  # 3. Calculate weight per class (inverse frequency)
  # 1.0 / count gives higher weight to rare classes
  class_weights = 1.0 / class_counts.float()

  # 4. Assign weight to every sample in the dataset
  sample_weights = class_weights[targets]

  # 5. Create sampler
  sampler = WeightedRandomSampler(
      weights=sample_weights,
      num_samples=len(sample_weights),
      replacement=True
  )
  return sampler


### 2. DataLoaders

In [3]:
def create_dataloaders(
    data_dir: str,
    batch_size: int = 32,
    num_workers: int = 2
) -> Dict[str, DataLoader]:
    """
    Initializes Datasets and DataLoaders.

    NOTE ON SPLITS: The Oxford 102 dataset has a peculiarity where the 'test'
    split is significantly larger (6000+) than the 'train' split (1000).
    Standard practice is to swap them or merge train+val.
    Here, we use 'test' for TRAINING as per user strategy.
    """

    transforms_dict = get_transforms(CONFIG["IMG_SIZE"])

    # 1. Define Datasets (Note the split swap logic)
    # We use the larger 'test' split for training to get better performance
    train_set = datasets.Flowers102(
        root=data_dir, split='test', download=True, transform=transforms_dict['train']
    )
    val_set = datasets.Flowers102(
        root=data_dir, split='val', download=True, transform=transforms_dict['val']
    )
    test_set = datasets.Flowers102(
        root=data_dir, split='train', download=True, transform=transforms_dict['val']
    )

    print(f"📊 Dataset Sizes - Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

    # 2. Define Sampler for Imbalance
    train_sampler = get_weighted_sampler(train_set)

    # 3. Create Loaders
    dataloaders = {
        'train': DataLoader(
            train_set,
            batch_size=batch_size,
            sampler=train_sampler, # Sampler and Shuffle are mutually exclusive
            num_workers=num_workers
        ),
        'val': DataLoader(
            val_set,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers
        ),
        'test': DataLoader(
            test_set,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers
        )
    }

    return dataloaders

In [4]:
# --- Execution ---
set_seed(CONFIG["SEED"])

# Generate the dictionary containing train/val/test loaders
dataloaders_dict = create_dataloaders(
    data_dir=CONFIG["DATA_DIR"],
    batch_size=CONFIG["BATCH_SIZE"],
    num_workers=CONFIG["NUM_WORKERS"]
)

print("✅ Data Pipeline Complete.")

🔒 Random Seed set to: 42
📊 Dataset Sizes - Train: 6149 | Val: 1020 | Test: 1020
✅ Data Pipeline Complete.



### 4. Model Setup (EfficientNet-B0)
* For fine-grained classification (distinguishing between specific flower species), standard ResNets are good, but EfficientNet or Vision Transformers (ViT) are often better because they capture subtle feature hierarchies more effectively.

* Faster training on Colab's free GPU and less risk of overfitting on a small dataset (102 classes, ~80 images each)

In [6]:
def set_parameter_requires_grad(model: nn.Module, feature_extracting: bool) -> None:
    """
    Helper function to freeze model parameters if feature extracting.
    """
    if feature_extracting:
        print("❄️ Freezing backbone layers...")
        for param in model.parameters():
            param.requires_grad = False
    else:
        print("🔥 Fine-tuning all layers...")

def get_model(num_classes: int = 102, feature_extract: bool = True) -> nn.Module:
    """
    Initializes an EfficientNet-B0 model for fine-tuning.

    Args:
        num_classes (int): Number of target classes (default: 102).
        feature_extract (bool): If True, freezes the backbone and only trains the head.

    Returns:
        nn.Module: The modified EfficientNet-B0 model.
    """
    print(f"🏗️ Initializing EfficientNet-B0 (Classes: {num_classes})...")

    # 1. Load Pre-trained Weights (ImageNet)
    model = models.efficientnet_b0(weights='DEFAULT')

    # 2. Freeze Backbone (if requested)
    set_parameter_requires_grad(model, feature_extract)

    # 3. Replace Classifier Head
    # EfficientNet classifier structure: Sequential(Dropout, Linear)
    # We need to replace the Linear layer at index 1

    # Verify the structure matches expectation (Safety check)
    if not isinstance(model.classifier[1], nn.Linear):
        raise ValueError("Unexpected EfficientNet architecture. Expected Linear layer at classifier[1].")

    num_ftrs = model.classifier[1].in_features

    # Replace with new Linear layer (automatically requires_grad=True)
    model.classifier[1] = nn.Linear(num_ftrs, num_classes)

    print("✅ Model Head Replaced. Ready for training.")
    return model


### 5. Early Stopping Utility

In [7]:
class EarlyStopping:
    """
    Implements Early Stopping to prevent overfitting.
    Stops training if validation loss doesn't improve after a given 'patience'.
    """
    def __init__(
        self,
        patience: int = 5,
        delta: float = 0.0,
        path: str = 'best_model.pth',
        verbose: bool = True
    ):
        """
        Args:
            patience (int): How many epochs to wait after last time val_loss improved.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
            path (str): Path for the checkpoint to be saved to.
            verbose (bool): If True, prints a message for each validation loss improvement.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.verbose = verbose

        self.counter: int = 0
        self.best_score: Optional[float] = None
        self.early_stop: bool = False
        self.val_loss_min: float = float('inf')

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        """
        Call method to update the early stopping status.

        Args:
            val_loss (float): The current epoch's validation loss.
            model (nn.Module): The model to save if performance improves.
        """
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)

        elif score < self.best_score + self.delta:
            # Score did not improve enough
            self.counter += 1
            if self.verbose:
                print(f'⏳ EarlyStopping counter: {self.counter} out of {self.patience}')

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            # Score improved
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss: float, model: nn.Module) -> None:
        """
        Saves model when validation loss decreases.
        """
        if self.verbose:
            print(f'✅ Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')

        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss


### 6. Training Loop Structure

In [8]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from typing import Dict, List, Tuple, Any

def train_model(
    model: nn.Module,
    dataloaders: Dict[str, DataLoader],
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    num_epochs: int = 25,
    patience: int = 5,
    device: torch.device = None
) -> Tuple[nn.Module, Dict[str, List[float]]]:
    """
    Trains the model with Early Stopping and History Tracking.

    Args:
        model: PyTorch model to train.
        dataloaders: Dictionary containing 'train' and 'val' DataLoaders.
        criterion: Loss function (e.g., CrossEntropyLoss).
        optimizer: Optimizer (e.g., AdamW).
        num_epochs: Maximum number of epochs.
        patience: Patience for Early Stopping.
        device: Device to train on (cuda/cpu).

    Returns:
        model: The trained model with best weights loaded.
        history: Dictionary containing loss and accuracy metrics.
    """

    # 1. Setup
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    since = time.time()

    # Initialize Early Stopping
    early_stopping = EarlyStopping(patience=patience, path='flower_model_best.pth', verbose=True)

    # Metric History
    history: Dict[str, List[float]] = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    print(f"🚀 Training started on {device} for {num_epochs} epochs...")

    # 2. Epoch Loop
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            # Use 'train' loader for training, 'val' loader for validation
            # Note: User's dataloader dict might use 'test' as 'train' (split swap),
            # so we ensure we access the correct key.
            # Assuming standard keys 'train' and 'val' exist in dataloaders dict.
            current_loader = dataloaders[phase]

            for inputs, labels in current_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history only if in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Calculate Epoch Metrics
            dataset_size = len(current_loader.dataset)
            epoch_loss = running_loss / dataset_size
            epoch_acc = running_corrects.double() / dataset_size

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Save History
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # 3. Early Stopping Check (only on validation phase)
            if phase == 'val':
                early_stopping(epoch_loss, model)

        # Check if we should stop
        if early_stopping.early_stop:
            print("🛑 Early stopping triggered.")
            break

        # Optional: Clear GPU cache at end of epoch
        torch.cuda.empty_cache()

    # 4. Wrap Up
    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Loss: {early_stopping.val_loss_min:.4f}')

    # Load the best model weights
    model.load_state_dict(torch.load('flower_model_best.pth'))

    return model, history

print("✅ Model Training Loop Refactored & Ready.")

✅ Model Training Loop Refactored & Ready.


### 7. Execution


In [10]:
# --- Experiment Configuration ---
# Defining these at the top is a Best Practice for reproducibility
HYPERPARAMS = {
    "NUM_CLASSES": 102,
    "LEARNING_RATE": 1e-4,   # Lower LR is better for fine-tuning
    "WEIGHT_DECAY": 1e-4,    # Regularization to prevent overfitting
    "NUM_EPOCHS": 20,
    "PATIENCE": 5,           # Early Stopping patience
    "FEATURE_EXTRACT": False # False = Fine-tune ALL layers (Standard for high accuracy)
}

In [11]:


def run_training_pipeline(dataloaders_dict):
    """
    Orchestrates the model initialization, setup, and training loop.
    """
    # 1. Device Setup
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"⚙️ Computation Device: {device}")
    if device.type == 'cuda':
        print(f"   ↳ GPU Name: {torch.cuda.get_device_name(0)}")

    # 2. Initialize Model
    # We use the refactored get_model function
    model_ft = get_model(
        num_classes=HYPERPARAMS["NUM_CLASSES"],
        feature_extract=HYPERPARAMS["FEATURE_EXTRACT"]
    )
    model_ft = model_ft.to(device)

    # 3. Define Loss & Optimizer
    criterion = nn.CrossEntropyLoss()

    # AdamW is generally superior to Adam for Computer Vision tasks due to better weight decay handling
    optimizer_ft = optim.AdamW(
        model_ft.parameters(),
        lr=HYPERPARAMS["LEARNING_RATE"],
        weight_decay=HYPERPARAMS["WEIGHT_DECAY"]
    )

    # 4. Execute Training
    print("\n🚀 Starting Training Loop...")
    model_ft, history = train_model(
        model=model_ft,
        dataloaders=dataloaders_dict,
        criterion=criterion,
        optimizer=optimizer_ft,
        num_epochs=HYPERPARAMS["NUM_EPOCHS"],
        patience=HYPERPARAMS["PATIENCE"],
        device=device
    )

    return model_ft, history

# --- Execution ---
# Ensure dataloaders_dict exists (from previous cell)
if 'dataloaders_dict' not in globals():
    # Re-create if missing (using the function from the previous refactor)
    dataloaders_dict = create_dataloaders(CONFIG["DATA_DIR"], CONFIG["BATCH_SIZE"])

# Run the experiment
trained_model, training_history = run_training_pipeline(dataloaders_dict)

⚙️ Computation Device: cuda:0
   ↳ GPU Name: Tesla T4
🏗️ Initializing EfficientNet-B0 (Classes: 102)...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 164MB/s]


🔥 Fine-tuning all layers...
✅ Model Head Replaced. Ready for training.

🚀 Starting Training Loop...
🚀 Training started on cuda:0 for 20 epochs...

Epoch 1/20
----------
Train Loss: 3.4983 Acc: 0.4352
Val Loss: 1.9824 Acc: 0.7382
✅ Validation loss decreased (inf --> 1.982447). Saving model...

Epoch 2/20
----------
Train Loss: 1.2576 Acc: 0.8411
Val Loss: 0.6636 Acc: 0.9059
✅ Validation loss decreased (1.982447 --> 0.663613). Saving model...

Epoch 3/20
----------
Train Loss: 0.4628 Acc: 0.9467
Val Loss: 0.3206 Acc: 0.9569
✅ Validation loss decreased (0.663613 --> 0.320575). Saving model...

Epoch 4/20
----------
Train Loss: 0.2285 Acc: 0.9712
Val Loss: 0.2424 Acc: 0.9578
✅ Validation loss decreased (0.320575 --> 0.242358). Saving model...

Epoch 5/20
----------
Train Loss: 0.1529 Acc: 0.9806
Val Loss: 0.1775 Acc: 0.9657
✅ Validation loss decreased (0.242358 --> 0.177504). Saving model...

Epoch 6/20
----------
Train Loss: 0.1022 Acc: 0.9844
Val Loss: 0.1500 Acc: 0.9657
✅ Validation los